In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_resources = spark.table("workspace.bronze.fhir_resources")

In [0]:
df_patients = bronze_resources.filter(col("resource_type") == "Patient")
#display(df_patients)

PATIENTS DATA

In [0]:
silver_patients = df_patients.select(
    col("resource.id").alias("patient_id"),
    col("resource.gender").alias("gender"),
    col("resource.birthDate").alias("DOB"),
    col("resource.address")[0]["city"].alias("City"),
    col("resource.address")[0]["state"].alias("State"),
    col("resource.deceasedDateTime").alias("deceased_datetime"),
    col("source_file_name"),
    col("ingest_timestamp")
)
#display(silver_patients)

ENCOUNTER DATA - Interaction between Patient and System

In [0]:
df_encounters = bronze_resources.filter(col("resource_type") == "Encounter")
#df_encounters.count()

In [0]:
silver_encounters = df_encounters.select(
    col("resource.id").alias("encounter_id"),
    regexp_replace(
        col("resource.subject.reference"),
        "urn:uuid:",
        ""
    ).alias("patient_id"),
    col("resource.status").alias("encounter_status"),
    col("resource.period.start").alias("encounter_start"),
    col("resource.period.end").alias("encounter_end"),
    col("resource.class.code").alias("encounter_class"),
    col("resource.serviceProvider.display").alias("encounter_provider"),
    col("source_file_name"),
    col("ingest_timestamp")
)
#silver_encounters.count()

In [0]:
df_conditions = bronze_resources.filter(col("resource_type") == "Condition")
display(df_conditions)

In [0]:
silver_conditions = df_conditions.select(
    col("resource.id").alias("condition_id"),
    regexp_replace(
        col("resource.subject.reference"),
        "urn:uuid:",
        ""
    ).alias("patient_id"),
    col("resource.code.coding")[0]["code"].alias("condition_code"),
    col("resource.code.coding")[0]["display"].alias("condition_description"),
    col("resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),
    col("resource.onsetDateTime").alias("condition_onset"),
    col("source_file_name"),
    col("ingest_timestamp")
)
#display(silver_conditions)

In [0]:
df_observations = bronze_resources.filter(col("resource_type") == "Observation")
#display(df_observations)

In [0]:
silver_observations = df_observations.select(
    col("resource.id").alias("observation_id"),
    regexp_replace(
        col("resource.subject.reference"),
        "urn:uuid:",
        ""
    ).alias("patient_id"),
    col("resource.code.coding")[0]["code"].alias("observation_code"),
    col("resource.code.coding")[0]["display"].alias("observation_description"),
    col("resource.valueQuantity.value").alias("observation_value"),
    col("resource.valueQuantity.unit").alias("observation_unit"),
    col("resource.effectiveDateTime").alias("observation_datetime"),
    col("source_file_name"),
    col("ingest_timestamp")
)
#display(silver_observations)

In [0]:
df_medications = bronze_resources.filter(
    col("resource_type") == "MedicationRequest"
)
silver_medications = df_medications.select(
    col("resource.id").alias("medication_request_id"),
    regexp_replace(
        col("resource.subject.reference"),
        "urn:uuid:",
        ""
    ).alias("patient_id"),
    col("resource.status").alias("medication_status"),
    col("resource.intent").alias("medication_intent"),
    col("resource.medicationCodeableConcept.coding")[0]["code"]
        .alias("medication_code"),
    col("resource.medicationCodeableConcept.coding")[0]["display"]
        .alias("medication_name"),
    col("resource.authoredOn")
        .alias("medication_authored_date"),
    col("source_file_name"),
    col("ingest_timestamp")
)
#display(silver_medications)

In [0]:
df_claim = bronze_resources.filter(col("resource_type") == "Claim")
display(df_claim)

In [0]:
silver_claim = df_claim.select(
    col("resource.id").alias("claim_id"),
    regexp_replace(
        col("resource.patient.reference"),
        "urn:uuid:",
        ""
    ).alias("patient_id"),
    regexp_replace(
        col("resource.item.encounter.reference")[0][0],
        "urn:uuid:",
        ""
    ).alias("encounter_id"),
    regexp_replace(
        col("resource.provider.reference"), "urn:uuid:", ""
    ).alias("provider_id"),
    col("resource.insurance.coverage.display")[0].alias("insurance_name"),
    col("resource.status").alias("claim_status"),  
    col("resource.provider.display").alias("provider_name"),
    get_json_object(col("resource.total"), "$.value").alias("total"),
    col("resource.priority.coding")[0]["code"].alias("priority_code")
)
display(silver_claim)

In [0]:
silver_patients.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.patients")
silver_encounters.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.encounters")
silver_conditions.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.conditions")
silver_observations.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.observations")
silver_medications.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.medications")
silver_claim.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.claims")